In [1]:
# On charge les 3 fichiers CSV générés par le pipeline d'ingestion
import pandas as pd
from pathlib import Path

dossier_data = Path("../data/raw")

# On récupère le fichier le plus récent pour chaque indicateur (au cas où il y en aurait plusieurs)
df_inflation = pd.read_csv(sorted(dossier_data.glob("inflation_senegal_*.csv"))[-1])
df_taux_change = pd.read_csv(sorted(dossier_data.glob("taux_change_uemoa_*.csv"))[-1])
df_pib = pd.read_csv(sorted(dossier_data.glob("pib_senegal_*.csv"))[-1])

# On affiche un aperçu pour vérifier que tout est bien chargé
print(df_inflation.head())
print(df_inflation.shape)

       period  value         indicateur  \
0  1960-01-01    NaN  inflation_senegal   
1  1961-01-01    NaN  inflation_senegal   
2  1962-01-01    NaN  inflation_senegal   
3  1963-01-01    NaN  inflation_senegal   
4  1964-01-01    NaN  inflation_senegal   

                                    description  
0  Taux d'inflation moyen annuel (IPC), Sénégal  
1  Taux d'inflation moyen annuel (IPC), Sénégal  
2  Taux d'inflation moyen annuel (IPC), Sénégal  
3  Taux d'inflation moyen annuel (IPC), Sénégal  
4  Taux d'inflation moyen annuel (IPC), Sénégal  
(65, 4)


In [2]:
# On ne garde que les lignes où la valeur existe (pas de trou)
df_inflation_propre = df_inflation[df_inflation['value'].notna()].copy()

# On trie par date, indispensable avant de calculer des retards ou moyennes mobiles
df_inflation_propre = df_inflation_propre.sort_values('period').reset_index(drop=True)

# Retard (lag) : la valeur de l'année précédente
df_inflation_propre['value_lag1'] = df_inflation_propre['value'].shift(1)

# Moyenne mobile sur 3 ans (lisse les à-coups)
df_inflation_propre['value_moy_mobile_3'] = df_inflation_propre['value'].rolling(window=3).mean()

# Taux de variation par rapport à l'année précédente (en %)
df_inflation_propre['value_variation_points'] = df_inflation_propre['value'].diff()

# On affiche pour vérifier
print(df_inflation_propre[['period', 'value', 'value_lag1', 'value_moy_mobile_3', 'value_variation_points']].head(10))

       period  value  value_lag1  value_moy_mobile_3  value_variation_points
0  1998-01-01   1.16         NaN                 NaN                     NaN
1  1999-01-01   0.81        1.16                 NaN                   -0.35
2  2000-01-01   0.74        0.81            0.903333                   -0.07
3  2001-01-01   2.98        0.74            1.510000                    2.24
4  2002-01-01   2.34        2.98            2.020000                   -0.64
5  2003-01-01  -0.03        2.34            1.763333                   -2.37
6  2004-01-01   0.50       -0.03            0.936667                    0.53
7  2005-01-01   1.71        0.50            0.726667                    1.21
8  2006-01-01   2.12        1.71            1.443333                    0.41
9  2007-01-01   5.86        2.12            3.230000                    3.74


In [3]:
def construire_features(df):
    """Nettoie une série et ajoute les features standards : lag, moyenne mobile, variation."""
    df_propre = df[df['value'].notna()].copy()
    df_propre = df_propre.sort_values('period').reset_index(drop=True)

    df_propre['value_lag1'] = df_propre['value'].shift(1)
    df_propre['value_moy_mobile_3'] = df_propre['value'].rolling(window=3).mean()
    df_propre['value_variation_points'] = df_propre['value'].diff()

    return df_propre

# On applique la même fonction aux 3 indicateurs
df_inflation_final = construire_features(df_inflation)
df_taux_change_final = construire_features(df_taux_change)
df_pib_final = construire_features(df_pib)

print(df_taux_change_final.tail())

        period   value         indicateur  \
60  2020-01-01  575.59  taux_change_uemoa   
61  2021-01-01  554.53  taux_change_uemoa   
62  2022-01-01  623.76  taux_change_uemoa   
63  2023-01-01  606.56  taux_change_uemoa   
64  2024-01-01  606.35  taux_change_uemoa   

                              description  value_lag1  value_moy_mobile_3  \
60  Taux de change XOF/USD, ensemble UMOA      585.67          572.280333   
61  Taux de change XOF/USD, ensemble UMOA      575.59          571.930000   
62  Taux de change XOF/USD, ensemble UMOA      554.53          584.626667   
63  Taux de change XOF/USD, ensemble UMOA      623.76          594.950000   
64  Taux de change XOF/USD, ensemble UMOA      606.56          612.223333   

    value_variation_points  
60                  -10.08  
61                  -21.06  
62                   69.23  
63                  -17.20  
64                   -0.21  


In [12]:
import pathlib
dossier = pathlib.Path("../data/raw")

df_pib_ci = pd.read_csv(dossier / "pib_cote_ivoire.csv")
df_inflation_ci = pd.read_csv(dossier / "inflation_cote_ivoire.csv")
df_pib_bf = pd.read_csv(dossier / "pib_burkina.csv")
df_inflation_bf = pd.read_csv(dossier / "inflation_burkina.csv")
df_pib_mali = pd.read_csv(dossier / "pib_mali.csv")
df_inflation_mali = pd.read_csv(dossier / "inflation_mali.csv")

print("6 fichiers rechargés")

6 fichiers rechargés


In [13]:
df_pib_ci_final = construire_features(df_pib_ci)
df_inflation_ci_final = construire_features(df_inflation_ci)
df_pib_bf_final = construire_features(df_pib_bf)
df_inflation_bf_final = construire_features(df_inflation_bf)
df_pib_mali_final = construire_features(df_pib_mali)
df_inflation_mali_final = construire_features(df_inflation_mali)

df_pib_ci_final.to_csv(dossier_processed / "pib_cote_ivoire_features.csv", index=False)
df_inflation_ci_final.to_csv(dossier_processed / "inflation_cote_ivoire_features.csv", index=False)
df_pib_bf_final.to_csv(dossier_processed / "pib_burkina_features.csv", index=False)
df_inflation_bf_final.to_csv(dossier_processed / "inflation_burkina_features.csv", index=False)
df_pib_mali_final.to_csv(dossier_processed / "pib_mali_features.csv", index=False)
df_inflation_mali_final.to_csv(dossier_processed / "inflation_mali_features.csv", index=False)

print("6 fichiers enrichis sauvegardés dans data/processed/")

6 fichiers enrichis sauvegardés dans data/processed/


In [4]:
from pathlib import Path

dossier_processed = Path("../data/processed")
dossier_processed.mkdir(parents=True, exist_ok=True)

df_inflation_final.to_csv(dossier_processed / "inflation_senegal_features.csv", index=False)
df_taux_change_final.to_csv(dossier_processed / "taux_change_uemoa_features.csv", index=False)
df_pib_final.to_csv(dossier_processed / "pib_senegal_features.csv", index=False)

print("Fichiers sauvegardés dans data/processed/")

Fichiers sauvegardés dans data/processed/


In [5]:
# Chargement et feature engineering des 3 nouveaux indicateurs
df_masse_monetaire = pd.read_csv(sorted(dossier_data.glob("masse_monetaire_senegal_*.csv"))[-1])
df_reserves = pd.read_csv(sorted(dossier_data.glob("reserves_change_senegal_*.csv"))[-1])
df_balance = pd.read_csv(sorted(dossier_data.glob("balance_commerciale_senegal_*.csv"))[-1])

df_masse_monetaire_final = construire_features(df_masse_monetaire)
df_reserves_final = construire_features(df_reserves)
df_balance_final = construire_features(df_balance)

print(df_balance_final.tail())

        period     value                   indicateur  \
23  1991-01-01  -87796.0  balance_commerciale_senegal   
24  1992-01-01  -96484.0  balance_commerciale_senegal   
25  1993-01-01 -107468.0  balance_commerciale_senegal   
26  1994-01-01 -128315.0  balance_commerciale_senegal   
27  1995-01-01 -136888.0  balance_commerciale_senegal   

                     description  value_lag1  value_moy_mobile_3  \
23  Balance commerciale, Sénégal    -73714.0       -79367.000000   
24  Balance commerciale, Sénégal    -87796.0       -85998.000000   
25  Balance commerciale, Sénégal    -96484.0       -97249.333333   
26  Balance commerciale, Sénégal   -107468.0      -110755.666667   
27  Balance commerciale, Sénégal   -128315.0      -124223.666667   

    value_variation_points  
23                -14082.0  
24                 -8688.0  
25                -10984.0  
26                -20847.0  
27                 -8573.0  


In [6]:
# Sauvegarde des 3 nouveaux indicateurs enrichis
df_masse_monetaire_final.to_csv(dossier_processed / "masse_monetaire_senegal_features.csv", index=False)
df_reserves_final.to_csv(dossier_processed / "reserves_change_senegal_features.csv", index=False)
df_balance_final.to_csv(dossier_processed / "balance_commerciale_senegal_features.csv", index=False)

print("3 nouveaux fichiers sauvegardés dans data/processed/")

3 nouveaux fichiers sauvegardés dans data/processed/


In [7]:
df_reserves = pd.read_csv(sorted(dossier_data.glob("reserves_change_senegal_*.csv"))[-1])
df_reserves_final = construire_features(df_reserves)
df_reserves_final.to_csv(dossier_processed / "reserves_change_senegal_features.csv", index=False)
print(df_reserves_final.tail())

        period      value               indicateur  \
49  2011-01-01   864853.5  reserves_change_senegal   
50  2012-01-01   879448.2  reserves_change_senegal   
51  2013-01-01   858465.5  reserves_change_senegal   
52  2014-01-01  1095559.0  reserves_change_senegal   
53  2015-01-01  1232922.6  reserves_change_senegal   

                                          description  value_lag1  \
49  Réserves de change (avoirs extérieurs nets), S...    987848.2   
50  Réserves de change (avoirs extérieurs nets), S...    864853.5   
51  Réserves de change (avoirs extérieurs nets), S...    879448.2   
52  Réserves de change (avoirs extérieurs nets), S...    858465.5   
53  Réserves de change (avoirs extérieurs nets), S...   1095559.0   

    value_moy_mobile_3  value_variation_points  
49        9.037534e+05               -122994.7  
50        9.107166e+05                 14594.7  
51        8.675891e+05                -20982.7  
52        9.444909e+05                237093.5  
53        1.062

In [8]:
# Chargement et feature engineering des 3 secteurs d'activité
df_agriculture = pd.read_csv(sorted(dossier_data.glob("secteur_agriculture_senegal_*.csv"))[-1])
df_industrie = pd.read_csv(sorted(dossier_data.glob("secteur_industrie_senegal_*.csv"))[-1])
df_services = pd.read_csv(sorted(dossier_data.glob("secteur_services_senegal_*.csv"))[-1])

df_agriculture_final = construire_features(df_agriculture)
df_industrie_final = construire_features(df_industrie)
df_services_final = construire_features(df_services)

df_agriculture_final.to_csv(dossier_processed / "secteur_agriculture_senegal_features.csv", index=False)
df_industrie_final.to_csv(dossier_processed / "secteur_industrie_senegal_features.csv", index=False)
df_services_final.to_csv(dossier_processed / "secteur_services_senegal_features.csv", index=False)

print("3 secteurs sauvegardés dans data/processed/")

3 secteurs sauvegardés dans data/processed/


In [14]:
comparaison = []

def ajouter_pays(pays, indicateur, df):
    for _, ligne in df.iterrows():
        comparaison.append({
            "pays": pays,
            "indicateur": indicateur,
            "annee": ligne["period"],
            "valeur": ligne["value"]
        })

ajouter_pays("Sénégal", "pib", df_pib)
ajouter_pays("Sénégal", "inflation", df_inflation)
ajouter_pays("Côte d'Ivoire", "pib", df_pib_ci_final)
ajouter_pays("Côte d'Ivoire", "inflation", df_inflation_ci_final)
ajouter_pays("Burkina Faso", "pib", df_pib_bf_final)
ajouter_pays("Burkina Faso", "inflation", df_inflation_bf_final)
ajouter_pays("Mali", "pib", df_pib_mali_final)
ajouter_pays("Mali", "inflation", df_inflation_mali_final)

df_comparaison = pd.DataFrame(comparaison)
df_comparaison.to_csv(dossier_processed / "comparaison_pays.csv", index=False)
df_comparaison.to_json(dossier_processed / "comparaison_pays.json", orient="records", indent=2)

print(df_comparaison.head(10))
print("\nFichiers exportés dans data/processed/")

      pays indicateur       annee  valeur
0  Sénégal        pib  1960-01-01   150.7
1  Sénégal        pib  1961-01-01   162.6
2  Sénégal        pib  1962-01-01   172.2
3  Sénégal        pib  1963-01-01   179.3
4  Sénégal        pib  1964-01-01   190.8
5  Sénégal        pib  1965-01-01   199.7
6  Sénégal        pib  1966-01-01   206.3
7  Sénégal        pib  1967-01-01   206.5
8  Sénégal        pib  1968-01-01   218.3
9  Sénégal        pib  1969-01-01   218.0

Fichiers exportés dans data/processed/
